# NB-03: Section Structure & Numbering Validator

Parses the full section hierarchy, compares against the expected numbering from the Excel content inventory, audits subsection label naming, and counts equation environments.

## Step 1 — Parse full section tree

In [1]:
import re
from pathlib import Path

TEX_FILE = "jmlr-hypatiax-paper-final.tex"
source = Path(TEX_FILE).read_text(encoding="utf-8")
lines  = source.splitlines()

SECT_RE = re.compile(
    r'\\((?:sub)*section)\*?\{([^}]+)\}.*?(?:\\label\{([^}]+)\})?',
    re.M)

sections = []
for m in SECT_RE.finditer(source):
    level = m.group(1).count("sub")  # 0=section, 1=sub, 2=subsub
    title = m.group(2).strip()
    label = m.group(3) or ""
    lineno = source[:m.start()].count("\n") + 1
    sections.append((level, title, label, lineno))

indent = {0: "", 1: "  ", 2: "    "}
for level, title, label, lineno in sections:
    lbl_str = f"  [\\label{{{label}}}]" if label else ""
    print(f"L{level} {indent[level]}{title}{lbl_str}  (line {lineno})")

L0 Introduction  (line 178)
L1   Contributions  (line 221)
L0 Related Work  (line 251)
L1   Symbolic Regression  (line 254)
L1   LLMs for Mathematical and Scientific Reasoning  (line 270)
L1   Equation Learners, Differentiable SR, and Uncertainty-Aware Methods  (line 283)
L1   Hybrid Symbolic--Neural Systems  (line 301)
L1   Extrapolation in Neural Networks  (line 314)
L0 Empirical Evidence of LLM Performance Across Domains  (line 324)
L1   Experimental Design  (line 329)
L1   Baseline Results: Bimodal Performance  (line 333)
L1   Failure Mode Taxonomy  (line 337)
L1   Case Studies  (line 348)
L1   Success Pattern Analysis  (line 356)
L1   Extrapolation Testing  (line 360)
L0 Theoretical Framework  (line 366)
L1   Formal Definitions  (line 369)
L1   Main Theoretical Result  (line 392)
L1   Implications for Discovery  (line 412)
L1   Functional Form Recovery and Inductive Bias Alignment  (line 416)
L0 Problem Formulation  (line 421)
L0 Benchmark Design  (line 475)
L1   Overview and Scop

## Step 2 — Expected vs. actual section structure

In [2]:
# Expected section numbering from Excel content inventory
# Format: (expected_number, title_fragment, expected_label)
EXPECTED = [
    (1, "Introduction",                          "sec:intro"),
    (2, "Related Work",                          "sec:related"),
    (3, "Empirical Evidence",                    "sec:llm_limitations"),
    (4, "Theoretical Framework",                 "sec:theory"),
    (5, "Problem Formulation",                   "sec:problem"),
    (6, "Benchmark Design",                      "sec:benchmark"),
    (7, "Methodology",                           "sec:method"),
    (8, "HypatiaX Architecture",                 "sec:architecture"),
    (9, "Experimental Setup",                    "sec:setup"),
    (10, "Results",                              "sec:results"),
    (11, "Discussion",                           "sec:discussion"),
    (12, "Conclusion",                           "sec:conclusion"),
]

print("Expected vs. Actual section structure (top-level only):")
print("-" * 80)
actual_sections = [(t, la, ln) for lv, t, la, ln in sections if lv == 0
                   and not t.startswith("*")]

for i, (exp_num, exp_frag, exp_label) in enumerate(EXPECTED):
    if i < len(actual_sections):
        act_title, act_label, act_line = actual_sections[i]
        title_ok  = exp_frag.lower() in act_title.lower()
        label_ok  = exp_label == act_label
        status_t  = "OK" if title_ok  else "MISMATCH"
        status_l  = "OK" if label_ok  else "MISMATCH"
        print(f"  §{exp_num:2d}  Title:  [{status_t}]  expected '{exp_frag}'  got '{act_title[:40]}'")
        print(f"        Label:  [{status_l}]  expected '{exp_label}'  got '{act_label}'")
    else:
        print(f"  §{exp_num:2d}  MISSING section")

Expected vs. Actual section structure (top-level only):
--------------------------------------------------------------------------------
  § 1  Title:  [OK]  expected 'Introduction'  got 'Introduction'
        Label:  [MISMATCH]  expected 'sec:intro'  got ''
  § 2  Title:  [OK]  expected 'Related Work'  got 'Related Work'
        Label:  [MISMATCH]  expected 'sec:related'  got ''
  § 3  Title:  [OK]  expected 'Empirical Evidence'  got 'Empirical Evidence of LLM Performance Ac'
        Label:  [MISMATCH]  expected 'sec:llm_limitations'  got ''
  § 4  Title:  [OK]  expected 'Theoretical Framework'  got 'Theoretical Framework'
        Label:  [MISMATCH]  expected 'sec:theory'  got ''
  § 5  Title:  [OK]  expected 'Problem Formulation'  got 'Problem Formulation'
        Label:  [MISMATCH]  expected 'sec:problem'  got ''
  § 6  Title:  [OK]  expected 'Benchmark Design'  got 'Benchmark Design'
        Label:  [MISMATCH]  expected 'sec:benchmark'  got ''
  § 7  Title:  [OK]  expected 'Methodo

## Step 3 — Subsection label audit

In [3]:
# Check subsection labels follow section numbering pattern
# e.g., subsections of §7 should have labels containing '7' or relevant names
print("Subsection label audit (sections 7-10):")
print("-" * 80)
relevant = [(lv, t, la, ln) for lv, t, la, ln in sections
            if lv == 1 and ln >= 0]

for level, title, label, lineno in relevant:
    if label:
        print(f"  {title[:50]:<52}  \\label{{{label}}}  (line {lineno})")
    else:
        print(f"  {title[:50]:<52}  [NO LABEL]  (line {lineno})")

Subsection label audit (sections 7-10):
--------------------------------------------------------------------------------
  Contributions                                         [NO LABEL]  (line 221)
  Symbolic Regression                                   [NO LABEL]  (line 254)
  LLMs for Mathematical and Scientific Reasoning        [NO LABEL]  (line 270)
  Equation Learners, Differentiable SR, and Uncertai    [NO LABEL]  (line 283)
  Hybrid Symbolic--Neural Systems                       [NO LABEL]  (line 301)
  Extrapolation in Neural Networks                      [NO LABEL]  (line 314)
  Experimental Design                                   [NO LABEL]  (line 329)
  Baseline Results: Bimodal Performance                 [NO LABEL]  (line 333)
  Failure Mode Taxonomy                                 [NO LABEL]  (line 337)
  Case Studies                                          [NO LABEL]  (line 348)
  Success Pattern Analysis                              [NO LABEL]  (line 356)
  Extrapol

## Step 4 — Equation environment inventory

In [4]:
# Count equations and check numbering environment types
EQ_ENVS  = re.findall(r'\\begin\{(equation|align|gather|multline)\*?\}', source)
LABEL_EQ = re.findall(r'\\begin\{equation\}.*?\\label\{(eq:[^}]+)\}', source, re.DOTALL)

print(f"Numbered equation environments: {len(EQ_ENVS)}")
print(f"  equation : {EQ_ENVS.count('equation')}")
print(f"  align    : {EQ_ENVS.count('align')}")
print(f"  gather   : {EQ_ENVS.count('gather')}")
print(f"  multline : {EQ_ENVS.count('multline')}")
print()
print(f"Equation labels (eq:*): {len(LABEL_EQ)}")
for k in LABEL_EQ:
    refs_in_text = source.count(f"\\ref{{{k}}}") + source.count(f"\\eqref{{{k}}}")
    print(f"  {k:<30}  referenced {refs_in_text}x")

Numbered equation environments: 14


  equation : 12
  align    : 2
  gather   : 0
  multline : 0

Equation labels (eq:*): 12
  eq:extrap                       referenced 1x
  eq:r2                           referenced 0x
  eq:llm_gen                      referenced 0x
  eq:r2_train_llm                 referenced 0x
  eq:r2_edge_llm                  referenced 0x
  eq:augmented_input              referenced 1x
  eq:trust                        referenced 0x
  eq:degradation                  referenced 0x
  eq:ood                          referenced 0x
  eq:routing                      referenced 0x
  eq:wllm                         referenced 1x
  eq:ensemble                     referenced 1x
